# Scryfall Tags: Multi-Label Classification  
__Objective:__ Since the scryfall tags are in essence a collection of multiple labels for each card, this problem is at it's core a mutli-label classification task.

## Packages and Data

In [ ]:
# packages

## connect project directory
import sys
from pathlib import Path
dir = str(Path(Path.cwd()).parents[0])
if dir not in sys.path:
    sys.path.append(dir)

## load from project directory
from src.data_gathering.scryfall_dataset import ScryfallDataset
from src.fine_tuning.modeling import FineTuneLLM

In [ ]:
# params

## data gathering
from src.config import BUILD_DATASET, TASK, DATASET_SIZE_N, TEST_SIZE_N
from src.config import MAX_INPUT_LENGTH, MAX_TARGET_LENGTH

## modeling
from src.config import MODEL_NAME
from src.config import BATCH_SIZE, LEARNING_RATE, WEIGHT_DECAY, NUM_EPOCHS
from src.config import GENERATION_MAX_LENGTH, GENERATION_NUM_BEAMS

In [ ]:
# get data
sf = ScryfallDataset(task = TASK)

## build dataset as needed
if BUILD_DATASET:
    sf.build_dataset(
        tag_path = '../reports/scryfall_tags.json',
        train_size_pct = 0.8,
        truncate_dataset = DATASET_SIZE_N,
        test_size_n = TEST_SIZE_N
    )

## load dataset
sf.load_hf_dataset(
    train_path = f'../data/scryfall_{TASK}_train.json',
    val_path = f'../data/scryfall_{TASK}_val.json',
    test_path = f'../data/scryfall_{TASK}_test.json'
)

Scryfall Tag Question Answering Dataset Built
	Train Records = 135
	Validation Records = 34
	Test Records = 10
	Records saved to...
		../data/scryfall_seq2seq_train.json
		../data/scryfall_seq2seq_val.json
		../data/scryfall_seq2seq_test.json
	NOTE: This method does not create the huggingface dataset object. Run load_dataset() for that.
Scryfall Tag Seq2Seq Dataset Loaded
	Train Records = 135
	Val Records = 34
	Test Records = 10
	Count Unique Tags = 0


## Modeling

In [ ]:
# from transformers import Trainer
# from torch.nn import BCEWithLogitsLoss

# class CustomTrainer(Trainer):
#     def __init__(self, pos_weights, *args, **kwargs):
#         super().__init__(*args, **kwargs)
#         self.pos_weights = pos_weights

#     def compute_loss(self, model, inputs, return_outputs = False):
#         labels = inputs.pop("labels")
#         outputs = model(**inputs) 
#         logits = outputs.logits

#         loss_fct = BCEWithLogitsLoss(pos_weight = self.pos_weights)
#         loss = loss_fct(logits, labels)

#         return (loss, outputs) if return_outputs else loss

In [ ]:
# fine tune the model
tagger = FineTuneLLM(
    model_name = MODEL_NAME,
    dataset = sf.dataset
)
tagger.prepare_data(
    max_input_length = MAX_INPUT_LENGTH,
    max_target_length = MAX_TARGET_LENGTH
)
tagger.train(
    batch_size = BATCH_SIZE,
    n_epochs = NUM_EPOCHS,
    learning_rate = LEARNING_RATE,
    weight_decay = WEIGHT_DECAY,
    generation_max_length = GENERATION_MAX_LENGTH,
    generation_num_beams = GENERATION_NUM_BEAMS
)

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Map:   0%|          | 0/135 [00:00<?, ? examples/s]

Map:   0%|          | 0/34 [00:00<?, ? examples/s]

Map:   0%|          | 0/10 [00:00<?, ? examples/s]

Map:   0%|          | 0/135 [00:00<?, ? examples/s]

Map:   0%|          | 0/34 [00:00<?, ? examples/s]

Map:   0%|          | 0/10 [00:00<?, ? examples/s]

TypeError: Seq2SeqTrainer.__init__() got an unexpected keyword argument 'tokenizer'